<a href="https://colab.research.google.com/github/DN575MGPU/programming/blob/main/%D0%9B%D0%B0%D0%B1%D0%B0_3_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Подготовленный CSV-файл содержит данные по детям (0-14 лет) и взрослым (15 лет и старше) за период 2000-2015 гг. в отношении на 1000 человек:

Класс,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,Возраст
болезни глаза и его придаточного аппарата,"31,9","32,4","33,5",33,"34,2","33,6","35,7","34,8",34,"33,5",33,"33,3","35,2",35,"34,7","33,3",Взрослые
болезни глаза и его придаточного аппарата,"4,67","4,81","5,32","5,13","5,43","5,58","5,5","5,68","5,7","5,68","5,81","5,87","6,12","6,03",6,"5,91",Дети
...


Предоставив пользователю выбрать 2 различных года для сравнения, сформируйте JSON-файл, который содержит абсолютные значения (для второго года) и изменения по количеству заболеваний в процентах (округление до 2-х знаков после запятой). Значения должны быть сгруппированы по категории Возраст и идти по убыванию.

In [12]:
# Задание task_03_04_08.
#
# Выполнил: Новожилкина Д. В.
# Группа: ЦИБ-251
# Вариант: 18


import csv
import json
import os

csv_filename = "medical_stats.csv"
export_filename = "output.json"


# Создается тестовый файл, если его нет

if not os.path.exists(csv_filename):
    # Числа с запятой берутся в кавычки ("34,7"), чтобы CSV понял это как одно число
    test_data = """Класс,2014,2015,Возраст
болезни глаза,"34,7","33,3",Взрослые
болезни глаза,"6,00","5,91",Дети
болезни дыхания,"220,7","225,3",Взрослые
болезни дыхания,"150,0","155,5",Дети"""

    # Открывает файл для записи ("w"). encoding="utf-8" гарантирует, что русские буквы сохранятся корректно
    with open(csv_filename, "w", encoding="utf-8") as f:
        f.write(test_data)
    print("Создан тестовый файл 'medical_stats.csv'")


# Читает данные из файла

def load_data(filename):
    data = []
    with open(filename, "r", encoding="utf-8") as f:  # Открывает файл в режиме чтения ("r")
        reader = csv.reader(f)                        # reader, читает CSV-файл построчно и разбивает строки на столбцы по запятым
        for row in reader:
            new_row = []
            for i, val in enumerate(row):
                # 0 - это название болезни, последний (len(row) - 1) - возраст.
                if i == 0 or i == len(row) - 1:
                    new_row.append(val)
                else:
                    # Заменяет запятую на точку и превращает в число (float)
                    try:
                        new_row.append(float(val.replace(",", ".")))
                    except ValueError:
                        new_row.append(val) # Если не вышло, сохраняет значение val как текст
            data.append(new_row)
    return data

# Обрабатывает данные и сохраняет в JSON

def export_data(filename, data, year_1, year_2):
    headers = data[0] # Первая строка - заголовки
    # Проверяет, есть ли запрошенные годы в заголовках
    if float(year_1) not in headers or float(year_2) not in headers:
        print(f"Ошибка: Год {year_1} или {year_2} не найден в файле!")
        return

    idx1 = headers.index(float(year_1))
    idx2 = headers.index(float(year_2))
  # Создает 4 пустых списка, куда будет складывать результаты для взрослых и детей
    adults_year2, children_year2 = [], []
    adults_change, children_change = [], []

    for row in data[1:]: # Проходит по всем строкам, кроме заголовка
        disease = row[0]
        age = row[-1].strip()
        val1 = row[idx1]
        val2 = row[idx2]

        # Считает процент изменения
        if val1 != 0:
          change = round(((val2 - val1) / val1) * 100, 2)
        else:
          change = 0.0

  # Создает маленькие словари вида {"болезни глаза": 33.3} и добавляет их в соответствующие списки
        if age == "Взрослые":
            adults_year2.append({disease: val2})
            adults_change.append({disease: change})
        elif age == "Дети":
            children_year2.append({disease: val2})
            children_change.append({disease: change})

    # Сортирует по убыванию (reverse=True) по числу внутри словаря
    adults_year2.sort(key=lambda x: list(x.values())[0], reverse=True) # reverse=True означает сортировать по убыванию
    children_year2.sort(key=lambda x: list(x.values())[0], reverse=True)
    adults_change.sort(key=lambda x: list(x.values())[0], reverse=True)
    children_change.sort(key=lambda x: list(x.values())[0], reverse=True)

    result = {
        "Второй год": {
            "Взрослые": adults_year2,
            "Дети": children_year2
        },
        "Изменения": {
            "Взрослые": adults_change,
            "Дети": children_change
        }
    }
    # Открывает выходной файл для записи. json.dump записывает словарь result в файл f.
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=4) # ensure_ascii=False разрешает записывать русские буквы напрямую, а не в виде кодов
                                                           # indent=4 делает файл с отступами в 4 пробела

# Запуск программы

data = load_data(csv_filename)

year_1, year_2 = 2014, 2015
print(f"Сравнивает годы: {year_1} и {year_2}\n")

export_data(export_filename, data, year_1, year_2)

print("Результат сохранен в 'output.json':\n")
# Открывает только что созданный JSON-файл в режиме чтения ("r"), считывает всё его содержимое методом .read()
with open(export_filename, "r", encoding="utf-8") as f:
    print(f.read())

Сравнивает годы: 2014 и 2015

Результат сохранен в 'output.json':

{
    "Второй год": {
        "Взрослые": [
            {
                "болезни системы кровообращения": 275.0
            },
            {
                "болезни органов дыхания": 225.3
            },
            {
                "болезни глаза и его придаточного аппарата": 33.3
            }
        ],
        "Дети": [
            {
                "болезни органов дыхания": 155.5
            },
            {
                "болезни системы кровообращения": 25.0
            },
            {
                "болезни глаза и его придаточного аппарата": 5.91
            }
        ]
    },
    "Изменения": {
        "Взрослые": [
            {
                "болезни органов дыхания": 2.08
            },
            {
                "болезни системы кровообращения": 1.85
            },
            {
                "болезни глаза и его придаточного аппарата": -4.03
            }
        ],
        "Дети": [
    